# 00. results.csv 한국어 데이터 해부

**목표:** 65,437행 × 114열의 원본을 보존하면서 중복·무정보 컬럼을 먼저 제거하고, 남은 102개 컬럼의
뜻과 응답값을 한국어 중심으로 바꾸어 데이터 구조·결측·분포·개별 응답을 꼼꼼히 살펴본다.

- 의미가 있는 영어 설문 응답은 한국어로 표시한다.
- 프로그래밍 언어, 제품, 서비스, 국가, 통화 같은 고유명사는 검색과 식별을 위해 원문을 유지한다.
- `;` 구분 다중응답은 항목별로 번역한다.
- 원본 DataFrame은 `raw_df`, 1차 정제 분석본은 `df`로 구분한다.

## 1. 데이터 불러오기

빈 문자열과 문자열 `NA`는 모두 미응답으로 처리한다. 아래 셀부터 순서대로 실행한다.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from eda_korean import COLUMN_KO, korean_view, translate_value

DATA_PATH = Path("results.csv")
assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
raw_df = pd.read_csv(DATA_PATH, na_values=["NA", ""], keep_default_na=True, low_memory=False)
assert len(COLUMN_KO) == len(raw_df.columns) == 114
assert set(COLUMN_KO) == set(raw_df.columns)

ADMIRATION_COLUMNS = [
    "LanguageAdmired", "DatabaseAdmired", "PlatformAdmired", "WebframeAdmired",
    "EmbeddedAdmired", "MiscTechAdmired", "ToolsTechAdmired",
    "NEWCollabToolsAdmired", "OfficeStackAsyncAdmired",
    "OfficeStackSyncAdmired", "AISearchDevAdmired",
]
DROP_COLUMNS = ["Check", *ADMIRATION_COLUMNS]
df = raw_df.drop(columns=DROP_COLUMNS).copy()

LEAKAGE_COLUMNS = [
    "AISearchDevHaveWorkedWith", "AISearchDevWantToWorkWith", "AISent", "AIComplex",
    "AIToolCurrently Using", "AIToolInterested in Using", "AIToolNot interested in Using",
    "AINextMuch more integrated", "AINextMore integrated", "AINextNo change",
    "AINextLess integrated", "AINextMuch less integrated", "AIThreat", "AIEthics",
    "AIBen", "AIAcc", "AIChallenges",
]
LOW_VALUE_COLUMNS = [
    "SurveyLength", "SurveyEase", "Currency", "CompTotal",
    "JobSatPoints_1", "JobSatPoints_4", "JobSatPoints_5", "JobSatPoints_6",
    "JobSatPoints_7", "JobSatPoints_8", "JobSatPoints_9",
    "JobSatPoints_10", "JobSatPoints_11",
]
WEAK_RELEVANCE_COLUMNS = [
    *[f"Knowledge_{i}" for i in range(1, 10)],
    *[f"Frequency_{i}" for i in range(1, 4)],
    "TimeSearching", "TimeAnswering", "JobSat",
]
MODEL_EXCLUDE_COLUMNS = LEAKAGE_COLUMNS + LOW_VALUE_COLUMNS + WEAK_RELEVANCE_COLUMNS
assert len(MODEL_EXCLUDE_COLUMNS) == len(set(MODEL_EXCLUDE_COLUMNS)) == 45
model_df = df.drop(columns=MODEL_EXCLUDE_COLUMNS).copy()

print(f"원본: {len(raw_df):,}행 × {raw_df.shape[1]}열")
print(f"1차 정제본: {len(df):,}행 × {df.shape[1]}열")
print(f"제거한 컬럼: {len(DROP_COLUMNS)}개")
print(f"모델링 후보본: {len(model_df):,}행 × {model_df.shape[1]}열")
print(f"모델 입력에서 제외한 컬럼: {len(MODEL_EXCLUDE_COLUMNS)}개")
print(f"정제본 메모리: {df.memory_usage(deep=True).sum() / 1024**2:,.2f} MB")

## 2. 1차 제거 기준과 전체 102개 컬럼 사전

이번 단계에서는 근거가 확실한 12개만 제외한다. `Check`는 유효값이 모두 `Apples`인 상수 컬럼이다.
11개 `*Admired` 컬럼은 전체 행에서 정확히 `사용 경험 ∩ 사용 희망`으로 재생성되므로 중복 파생값이다.
AI 후속 문항과 의미가 낮은 컬럼은 탐색용 `df`에는 보존하고 모델 입력 후보 `model_df`에서만 제외한다.

아래 표에서 남은 컬럼의 원본명·한국어명·자료형·결측·고유값 수·대표값을 확인한다.

## 2-1. 목표변수 `AISelect` 분포와 클래스 불균형

`AISelect`는 AI 도구를 현재 사용하는지, 곧 사용할 계획인지, 사용할 계획이 없는지를 나타내는 3범주 목표변수다.
결측은 모델 학습에서 제외하되, 특정 응답자 집단에 집중되는지 별도로 확인한다.

In [ ]:
TARGET = "AISelect"
TARGET_LABELS = {
    "Yes": "사용 중",
    "No, but I plan to soon": "곧 사용할 계획",
    "No, and I don't plan to": "사용할 계획 없음",
}

target_counts = df[TARGET].value_counts(dropna=False)
valid_target = df[TARGET].dropna()
valid_counts = valid_target.value_counts()
target_distribution = pd.DataFrame({
    "원본 범주": valid_counts.index,
    "한국어 범주": [TARGET_LABELS[v] for v in valid_counts.index],
    "응답 수": valid_counts.values,
    "유효 응답 대비(%)": (valid_counts.values / len(valid_target) * 100).round(2),
    "전체 대비(%)": (valid_counts.values / len(df) * 100).round(2),
})
display(target_distribution)

missing_count = df[TARGET].isna().sum()
majority_count = valid_counts.max()
minority_count = valid_counts.min()
print(f"전체 데이터: {len(df):,}건")
print(f"유효 목표값: {len(valid_target):,}건 ({len(valid_target) / len(df) * 100:.2f}%)")
print(f"목표값 결측: {missing_count:,}건 ({missing_count / len(df) * 100:.2f}%)")
print(f"최다/최소 클래스 비율: {majority_count / minority_count:.2f}:1")
print(f"다수 클래스 기준 정확도: {majority_count / len(valid_target) * 100:.2f}%")

### 클래스 불균형 해석

최다 클래스와 최소 클래스 차이는 약 4.48배다. 모델 학습이 불가능할 정도의 극단적 불균형은 아니지만,
정확도만 보면 소수 클래스인 `곧 사용할 계획`을 제대로 예측하지 못해도 성능이 좋아 보일 수 있다.
따라서 분할 시 계층화(`stratify`)를 적용하고 Macro F1, Balanced Accuracy, 클래스별 정밀도·재현율을 함께 평가한다.
첫 기준 모델은 모든 응답을 최다 클래스인 `사용 중`으로 예측했을 때의 정확도 61.84%다.

In [ ]:
branch_target = pd.crosstab(
    df["MainBranch"].map(translate_value),
    df[TARGET].map(TARGET_LABELS),
    normalize="index",
).mul(100).round(2)
branch_missing = df.groupby("MainBranch")[TARGET].agg(
    전체="size", 유효값="count"
)
branch_missing["결측률(%)"] = ((branch_missing["전체"] - branch_missing["유효값"]) / branch_missing["전체"] * 100).round(2)
print("개발 활동 유형별 AISelect 유효 응답 분포(%)")
display(branch_target)
print("개발 활동 유형별 목표값 결측률")
display(branch_missing.sort_values("결측률(%)", ascending=False))

In [ ]:
column_dictionary = pd.DataFrame({
    "순번": range(1, len(df.columns) + 1),
    "원본 컬럼명": df.columns,
    "한국어 컬럼명": [COLUMN_KO[c] for c in df.columns],
    "자료형": df.dtypes.astype(str).values,
    "유효값 수": df.notna().sum().values,
    "미응답 수": df.isna().sum().values,
    "미응답률(%)": df.isna().mean().mul(100).round(2).values,
    "고유값 수": df.nunique(dropna=True).values,
    "대표값(한국어 표시)": [
        translate_value(df[c].dropna().iloc[0]) if df[c].notna().any() else "미응답만 존재"
        for c in df.columns
    ],
})
display(column_dictionary)

## 3. 한국어 전체 데이터 미리보기

`df_ko`는 원본 `df`의 표시용 복사본이다. 65,437행 전체를 한꺼번에 복사하면 메모리를 많이 쓰므로
기본값은 앞 100행이며, 필요하면 `VIEW_ROWS`를 늘린다.

In [ ]:
VIEW_ROWS = 100
df_ko = korean_view(df.head(VIEW_ROWS))
display(df_ko)

## 4. 원하는 컬럼 하나를 완전히 뜯어보기

`COLUMN`에 원본 컬럼명 또는 한국어 컬럼명을 넣으면 결측률, 모든 고유 응답, 빈도와 비율을 확인한다.
다중응답 컬럼은 `;` 안의 선택지를 각각 분리해 별도 순위도 보여준다.

In [ ]:
COLUMN = "AISelect"  # 예: "AISelect", "AI 도구 사용 여부", "DevType"

reverse_columns = {korean: original for original, korean in COLUMN_KO.items()}
original_column = reverse_columns.get(COLUMN, COLUMN)
if original_column not in df.columns:
    raise KeyError(f"존재하지 않는 컬럼입니다: {COLUMN}")

series = df[original_column]
counts = series.fillna("<MISSING>").value_counts(dropna=False)
detail = pd.DataFrame({
    "원본 응답값": counts.index,
    "한국어 표시값": [translate_value(v) for v in counts.index],
    "응답 수": counts.values,
    "전체 대비(%)": (counts.values / len(df) * 100).round(2),
})
print(f"[{original_column}] {COLUMN_KO[original_column]}")
print(f"자료형={series.dtype}, 고유값={series.nunique(dropna=True):,}, 미응답={series.isna().sum():,} ({series.isna().mean()*100:.2f}%)")
display(detail)

if series.dropna().astype(str).str.contains(";", regex=False).any():
    choices = series.dropna().astype(str).str.split(";").explode().str.strip()
    choices = choices[choices.ne("")].value_counts()
    multi_detail = pd.DataFrame({
        "원본 선택지": choices.index,
        "한국어 표시값": [translate_value(v) for v in choices.index],
        "선택 횟수": choices.values,
        "응답자 대비 선택률(%)": (choices.values / len(df) * 100).round(2),
    })
    print("다중응답 선택지별 집계")
    display(multi_detail)

## 5. 모든 범주형 컬럼의 값 목록

고유값이 너무 많은 자유수치·ID 컬럼은 제외하고, 고유값 200개 이하인 컬럼의 모든 응답값을
컬럼별로 펼친 카탈로그를 만든다. 표 검색 기능으로 한국어명이나 응답을 찾을 수 있다.

In [ ]:
catalog_rows = []
for column in df.columns:
    if df[column].nunique(dropna=True) > 200:
        continue
    counts = df[column].fillna("<MISSING>").value_counts(dropna=False)
    for original_value, count in counts.items():
        catalog_rows.append({
            "원본 컬럼명": column,
            "한국어 컬럼명": COLUMN_KO[column],
            "원본 응답값": original_value,
            "한국어 표시값": translate_value(original_value),
            "응답 수": count,
            "비율(%)": round(count / len(df) * 100, 2),
        })
value_catalog = pd.DataFrame(catalog_rows)
print(f"범주값 카탈로그: {len(value_catalog):,}개 행")
display(value_catalog)

## 6. 결측과 자료형 한눈에 보기

설문 분기 때문에 생긴 구조적 결측이 포함되므로 결측이 많다고 바로 삭제하지 않는다.

In [ ]:
missing_view = column_dictionary.sort_values("미응답률(%)", ascending=False)
display(missing_view[["원본 컬럼명", "한국어 컬럼명", "자료형", "미응답 수", "미응답률(%)", "고유값 수"]])

## 7. 수치형 컬럼 전체 요약

개수·평균·표준편차·최솟값·사분위수·최댓값을 전치해 컬럼별로 비교한다.
`CompTotal`은 국가별 통화가 다르므로 국가 간 비교에는 `ConvertedCompYearly`를 사용한다.

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns
numeric_summary = df[numeric_columns].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
numeric_summary.insert(0, "한국어 컬럼명", [COLUMN_KO[c] for c in numeric_summary.index])
display(numeric_summary)

## 8. 응답자 한 명의 114개 답변 세로로 보기

가로 114열 대신 한 응답자의 모든 항목을 세로로 펼쳐 원문과 한국어 표시를 나란히 본다.
`ROW_NUMBER`는 0부터 시작한다.

In [ ]:
ROW_NUMBER = 0
if not 0 <= ROW_NUMBER < len(df):
    raise IndexError(f"ROW_NUMBER는 0~{len(df)-1:,} 범위여야 합니다.")
respondent = pd.DataFrame({
    "원본 컬럼명": df.columns,
    "한국어 컬럼명": [COLUMN_KO[c] for c in df.columns],
    "원본 응답": df.iloc[ROW_NUMBER].values,
    "한국어 표시": [translate_value(v) for v in df.iloc[ROW_NUMBER].values],
})
print(f"행 번호: {ROW_NUMBER:,} / 응답 ID: {df.iloc[ROW_NUMBER]['ResponseId']}")
display(respondent)